# 3. Deterministic Modeling

This notebook covers the preliminary studies required to justify the final deterministic model design before production training. The dataset has already been cleaned and preprocessed.

**Steps:**
1. **Data Loading and Splitting:** Load the preprocessed dataset and split into train/test sets.
2. **Architecture Comparison:** Ablation study on depth, width, dropout, and activation on a global model.
3. **Segmentation Comparison:** Compare 1 to 5 price segments to justify the segmentation strategy.
4. **Final Training:** Train and evaluate the deterministic models for the selected configuration.
5. **Artifact Export:** Save the trained segment models for downstream use.

In [ ]:
import pandas as pd
import numpy as np
import tensorflow as tf
from sklearn.model_selection import train_test_split
from tensorflow.keras import layers, models, callbacks, optimizers, losses
import matplotlib.pyplot as plt
import seaborn as sns
import os
import time
import warnings

warnings.filterwarnings('ignore')
tf.get_logger().setLevel('ERROR')

GLOBAL_SEED = 42
tf.random.set_seed(GLOBAL_SEED)
np.random.seed(GLOBAL_SEED)

## Data Loading and Splitting

In [ ]:
df = pd.read_csv('../../data/madrid/listings_experimental_processed.csv')

X = df.drop('price', axis=1).values.astype('float32')
y = df['price'].values.astype('float32')

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Train features shape: {X_train.shape}")
print(f"Test features shape: {X_test.shape}")

## Architecture Comparison

We perform ablation experiments on the global dataset (0-249€) to study the effect of four hyperparameter axes before applying segmentation:
- **A1: Depth** — 2, 3, and 4 hidden layers
- **A2: Width** — small, medium, and large layer widths
- **A3: Dropout** — rates 0.0, 0.1, 0.2, and 0.5
- **A4: Activation** — sigmoid, tanh, and relu

In [ ]:
ARCH_DIR = '../architecture_comparison'
os.makedirs(ARCH_DIR, exist_ok=True)

def build_model_custom(input_dim, hidden_layers, dropout_rate, activation):
    model = models.Sequential()
    model.add(layers.Input(shape=(input_dim,)))
    for units in hidden_layers:
        model.add(layers.Dense(units, activation=activation))
        if dropout_rate > 0:
            model.add(layers.Dropout(dropout_rate))
    model.add(layers.Dense(1))
    model.compile(optimizer='adam', loss='huber', metrics=['mae'])
    return model

def train_and_evaluate(config_name, hidden_layers, dropout_rate, activation):
    model = build_model_custom(X_train.shape[1], hidden_layers, dropout_rate, activation)
    
    start = time.time()
    history = model.fit(
        X_train, y_train,
        epochs=1000, batch_size=32, validation_split=0.2, verbose=0,
        callbacks=[callbacks.EarlyStopping(patience=20, restore_best_weights=True)]
    )
    training_time = time.time() - start
    
    test_mae  = model.evaluate(X_test, y_test, verbose=0)[1]
    train_mae = model.evaluate(X_train, y_train, verbose=0)[1]
    val_mae   = min(history.history['val_mae'])
    
    y_pred = model.predict(X_test, verbose=0).flatten()
    rmse   = np.sqrt(np.mean((y_test - y_pred) ** 2))
    mape   = np.mean(np.abs((y_test - y_pred) / y_test)) * 100
    
    print(f"  {config_name}  MAE: €{test_mae:.2f} | MAPE: {mape:.2f}% | Params: {model.count_params():,}")
    
    return {
        'config_name': config_name,
        'hidden_layers': str(hidden_layers),
        'dropout_rate': float(dropout_rate),
        'activation': activation,
        'test_mae': float(test_mae),
        'test_rmse': float(rmse),
        'test_mape': float(mape),
        'train_mae': float(train_mae),
        'val_mae': float(val_mae),
        'overfitting_gap': float(train_mae - val_mae),
        'n_params': int(model.count_params()),
        'training_time_sec': float(training_time),
        'epochs_trained': len(history.history['loss'])
    }

all_arch_results = {}

print("A1: Depth")
all_arch_results['depth'] = [
    train_and_evaluate("2 layers", [128, 64],         0.2, 'relu'),
    train_and_evaluate("3 layers", [128, 64, 32],     0.2, 'relu'),
    train_and_evaluate("4 layers", [128, 64, 32, 16], 0.2, 'relu'),
]

print("\nA2: Width")
all_arch_results['width'] = [
    train_and_evaluate("small",  [64, 32, 16],   0.2, 'relu'),
    train_and_evaluate("medium", [128, 64, 32],  0.2, 'relu'),
    train_and_evaluate("large",  [256, 128, 64], 0.2, 'relu'),
]

print("\nA3: Dropout")
all_arch_results['dropout'] = [
    train_and_evaluate(f"dropout {drop:.1f}", [128, 64, 32], drop, 'relu')
    for drop in [0.0, 0.1, 0.2, 0.5]
]

print("\nA4: Activation")
all_arch_results['activation'] = [
    train_and_evaluate(act, [128, 64, 32], 0.2, act)
    for act in ['sigmoid', 'tanh', 'relu']
]

# Save individual CSVs and summary
all_rows = []
for exp_name, res_list in all_arch_results.items():
    df_exp = pd.DataFrame(res_list)
    df_exp.to_csv(f'{ARCH_DIR}/ablation_{exp_name}.csv', index=False)
    for r in res_list:
        r['experiment'] = exp_name
        all_rows.append(r)

summary_df = pd.DataFrame(all_rows)
summary_df.to_csv(f'{ARCH_DIR}/ablation_summary.csv', index=False)
print(f"\nResults saved to: {ARCH_DIR}/")
print(summary_df[['experiment','config_name','test_mae','test_mape','n_params','epochs_trained']].to_string(index=False))

In [ ]:
sns.set_style("whitegrid")
LABEL_SIZE = 11
TITLE_SIZE = 13
LABEL_COLOR = 'black'

fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# A1: Depth
ax = axes[0, 0]
depth_results = all_arch_results['depth']
configs = [r['config_name'] for r in depth_results]
maes    = [r['test_mae']    for r in depth_results]
x = np.arange(len(configs))
bars = ax.bar(x, maes, color='#6200EA', alpha=0.85, width=0.5)
for bar, val in zip(bars, maes):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.05,
            f'€{val:.2f}', ha='center', va='bottom', fontsize=10, color=LABEL_COLOR)
spread = max(maes) - min(maes)
ax.set_ylim(min(maes) - spread * 0.5, max(maes) + spread * 1)
ax.set_xlabel('Configuration', fontsize=LABEL_SIZE, color=LABEL_COLOR)
ax.set_ylabel('Test MAE (€)',   fontsize=LABEL_SIZE, color=LABEL_COLOR)
ax.set_title('A1: Depth Comparison', fontsize=TITLE_SIZE, fontweight='bold', color=LABEL_COLOR)
ax.set_xticks(x); ax.set_xticklabels(configs)
ax.grid(True, alpha=0.3, axis='y')
ax.tick_params(colors=LABEL_COLOR)

# A2: Width
ax = axes[0, 1]
width_results = all_arch_results['width']
configs = [r['config_name'] for r in width_results]
maes    = [r['test_mae']    for r in width_results]
params  = [r['n_params']    for r in width_results]
x = np.arange(len(configs))
bars = ax.bar(x, maes, color='#4ECDC4', alpha=0.85, width=0.5)
for bar, val, p in zip(bars, maes, params):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.05,
            f'€{val:.2f}\n({p:,} params)', ha='center', va='bottom', fontsize=9, color=LABEL_COLOR)
spread = max(maes) - min(maes)
ax.set_ylim(min(maes) - spread * 0.5, max(maes) + spread * 1)
ax.set_xlabel('Configuration', fontsize=LABEL_SIZE, color=LABEL_COLOR)
ax.set_ylabel('Test MAE (€)',   fontsize=LABEL_SIZE, color=LABEL_COLOR)
ax.set_title('A2: Width Comparison', fontsize=TITLE_SIZE, fontweight='bold', color=LABEL_COLOR)
ax.set_xticks(x); ax.set_xticklabels(configs)
ax.grid(True, alpha=0.3, axis='y')
ax.tick_params(colors=LABEL_COLOR)

# A3: Dropout
ax = axes[1, 0]
dropout_results = all_arch_results['dropout']
dropout_rates = [r['dropout_rate'] for r in dropout_results]
train_maes    = [r['train_mae']    for r in dropout_results]
test_maes     = [r['test_mae']     for r in dropout_results]
ax.plot(dropout_rates, train_maes, marker='o', label='Train MAE', linewidth=2, color='#2196F3')
ax.plot(dropout_rates, test_maes,  marker='s', label='Test MAE',  linewidth=2, color='#FF9800')
ax.set_xlabel('Dropout Rate',  fontsize=LABEL_SIZE, color=LABEL_COLOR)
ax.set_ylabel('MAE (€)',       fontsize=LABEL_SIZE, color=LABEL_COLOR)
ax.set_title('A3: Dropout Comparison', fontsize=TITLE_SIZE, fontweight='bold', color=LABEL_COLOR)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
ax.tick_params(colors=LABEL_COLOR)

# A4: Activation
ax = axes[1, 1]
act_results = all_arch_results['activation']
configs = [r['activation'] for r in act_results]
maes    = [r['test_mae']   for r in act_results]
x = np.arange(len(configs))
bars = ax.bar(x, maes, color='#FF6B6B', alpha=0.85, width=0.5)
for bar, val in zip(bars, maes):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.05,
            f'€{val:.2f}', ha='center', va='bottom', fontsize=10, color=LABEL_COLOR)
spread = max(maes) - min(maes)
ax.set_ylim(min(maes) - spread * 0.5, max(maes) + spread * 1)
ax.set_xlabel('Activation Function', fontsize=LABEL_SIZE, color=LABEL_COLOR)
ax.set_ylabel('Test MAE (€)',        fontsize=LABEL_SIZE, color=LABEL_COLOR)
ax.set_title('A4: Activation Comparison', fontsize=TITLE_SIZE, fontweight='bold', color=LABEL_COLOR)
ax.set_xticks(x); ax.set_xticklabels(configs)
ax.grid(True, alpha=0.3, axis='y')
ax.tick_params(colors=LABEL_COLOR)

plt.tight_layout()
plt.savefig(f'{ARCH_DIR}/ablation_results.png', dpi=300, bbox_inches='tight')
plt.show()

## Segmentation Comparison

Using a fixed architecture (3 layers: 128→64→32, dropout 0.2, ReLU), we compare five segmentation strategies over the 0-249€ range and measure weighted MAE across 1, 2, 3, 4, and 5 independent models.

In [ ]:
SEG_DIR = '../segmentation_comparison'
os.makedirs(SEG_DIR, exist_ok=True)

ARCH_FOR_SEG = {'hidden_layers': [128, 64, 32], 'dropout_rate': 0.2, 'activation': 'relu'}

SEGMENTATION_STRATEGIES = {
    '1_segment': [{'name': 'All', 'min': 0, 'max': 249}],
    '2_segments': [
        {'name': 'Low',  'min': 0,   'max': 128},
        {'name': 'High', 'min': 128, 'max': 249}
    ],
    '3_segments': [
        {'name': 'Low',    'min': 0,   'max': 110},
        {'name': 'Medium', 'min': 110, 'max': 164},
        {'name': 'High',   'min': 164, 'max': 249}
    ],
    '4_segments': [
        {'name': 'Low',         'min': 0,   'max': 95},
        {'name': 'Medium',      'min': 95,  'max': 128},
        {'name': 'Medium_High', 'min': 128, 'max': 182},
        {'name': 'High',        'min': 182, 'max': 249}
    ],
    '5_segments': [
        {'name': 'Low',         'min': 0,   'max': 79},
        {'name': 'Low_Medium',  'min': 79,  'max': 110},
        {'name': 'Medium',      'min': 110, 'max': 150},
        {'name': 'Medium_High', 'min': 150, 'max': 193},
        {'name': 'High',        'min': 193, 'max': 249}
    ]
}

def train_segment(x_tr, y_tr, x_te, y_te, segment_name):
    model = build_model_custom(x_tr.shape[1], **ARCH_FOR_SEG)
    start = time.time()
    history = model.fit(
        x_tr, y_tr, epochs=1000, batch_size=32, validation_split=0.2, verbose=0,
        callbacks=[callbacks.EarlyStopping(patience=20, restore_best_weights=True)]
    )
    training_time = time.time() - start
    test_mae = model.evaluate(x_te, y_te, verbose=0)[1]
    y_pred   = model.predict(x_te, verbose=0).flatten()
    rmse     = np.sqrt(np.mean((y_te - y_pred) ** 2))
    mape     = np.mean(np.abs((y_te - y_pred) / y_te)) * 100
    print(f"    {segment_name:15s}  MAE: €{test_mae:.2f} | MAPE: {mape:.2f}% | Time: {training_time:.1f}s")
    return {
        'segment': segment_name,
        'n_train': len(y_tr), 'n_test': len(y_te),
        'test_mae': float(test_mae), 'test_rmse': float(rmse), 'test_mape': float(mape),
        'training_time_sec': float(training_time),
        'epochs_trained': len(history.history['loss'])
    }

all_seg_results = {}
summary_rows = []

for strategy_name, segments in SEGMENTATION_STRATEGIES.items():
    label = strategy_name.replace('_', ' ')
    print(f"\nStrategy: {label}")
    segment_results = []

    for seg in segments:
        mask_tr = (y_train > seg['min']) & (y_train <= seg['max'])
        mask_te = (y_test  > seg['min']) & (y_test  <= seg['max'])
        if mask_tr.sum() < 50 or mask_te.sum() < 10:
            print(f"  Skipping {seg['name']}: insufficient samples")
            continue
        result = train_segment(X_train[mask_tr], y_train[mask_tr], X_test[mask_te], y_test[mask_te], seg['name'])
        segment_results.append(result)

    all_seg_results[strategy_name] = segment_results

    total_test = sum(r['n_test'] for r in segment_results)
    w_mae  = sum(r['test_mae']  * r['n_test'] for r in segment_results) / total_test
    w_mape = sum(r['test_mape'] * r['n_test'] for r in segment_results) / total_test
    avg_time = sum(r['training_time_sec'] for r in segment_results) / len(segment_results)

    print(f"  Weighted MAE: €{w_mae:.2f} | Weighted MAPE: {w_mape:.2f}%")

    pd.DataFrame(segment_results).to_csv(f'{SEG_DIR}/{strategy_name}_results.csv', index=False)

    summary_rows.append({
        'strategy': label,
        'n_segments': len(segment_results),
        'weighted_mae': w_mae,
        'weighted_mape': w_mape,
        'total_test_samples': total_test,
        'avg_training_time_sec': avg_time
    })

summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv(f'{SEG_DIR}/segmentation_summary.csv', index=False)
print("\n", summary_df.to_string(index=False))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

n_segs = summary_df['n_segments'].tolist()
maes   = summary_df['weighted_mae'].tolist()

# Plot 1: Weighted MAE vs n_segments
ax = axes[0]
ax.plot(n_segs, maes, marker='o', linewidth=2, markersize=10, color='#6200EA')
for n, mae in zip(n_segs, maes):
    ax.annotate(f'€{mae:.2f}', (n, mae), textcoords="offset points", xytext=(0,10), ha='center')
ax.set_xlabel('Number of Segments', fontsize=12)
ax.set_ylabel('Weighted MAE (€)', fontsize=12)
ax.set_title('Segmentation Impact on Performance', fontsize=14, fontweight='bold')
ax.set_xticks(n_segs); ax.grid(True, alpha=0.3)

# Plot 2: Relative improvement vs baseline (1 segment)
ax = axes[1]
baseline_mae = summary_df[summary_df['n_segments'] == 1]['weighted_mae'].values[0]
improvements = [(baseline_mae - mae) / baseline_mae * 100 for mae in maes]
colors = ['#FF6B6B' if imp < 0 else '#4ECDC4' for imp in improvements]
ax.bar(n_segs, improvements, color=colors, alpha=0.8)
ax.axhline(y=0, color='black', linestyle='--', linewidth=1)
for n, imp in zip(n_segs, improvements):
    ax.annotate(f'{imp:.1f}%', (n, imp), textcoords="offset points",
                xytext=(0, 5 if imp >= 0 else -15), ha='center')
ax.set_xlabel('Number of Segments', fontsize=12)
ax.set_ylabel('MAE Improvement vs 1 Segment (%)', fontsize=12)
ax.set_title('Relative Performance Gain', fontsize=14, fontweight='bold')
ax.set_xticks(n_segs); ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig(f'{SEG_DIR}/segmentation_results.png', dpi=300, bbox_inches='tight')
plt.show()

## Final Training & Evaluation

To verify everything operates correctly before passing models to production, we train and evaluate the selected 5 deterministic segments locally.

In [ ]:
# Architecture and segmentation for production
ARCH = {'hidden_layers': [128, 64, 32], 'dropout_rate': 0.2, 'activation': 'relu'}

SEGMENTS = [
    {'name': 'Low',         'min': 0,   'max': 79},
    {'name': 'Low_Medium',  'min': 79,  'max': 110},
    {'name': 'Medium',      'min': 110, 'max': 150},
    {'name': 'Medium_High', 'min': 150, 'max': 193},
    {'name': 'High',        'min': 193, 'max': 249}
]

TRAINING_RESULTS_DIR = '../training'
os.makedirs(TRAINING_RESULTS_DIR, exist_ok=True)

final_deterministic_models = {}
training_records = []

for seg in SEGMENTS:
    name = seg['name']
    print(f"Training {name} (€{seg['min']}-{seg['max']})")
    
    mask_tr = (y_train > seg['min']) & (y_train <= seg['max'])
    mask_te = (y_test  > seg['min']) & (y_test  <= seg['max'])
    
    x_tr_seg, y_tr_seg = X_train[mask_tr], y_train[mask_tr]
    x_te_seg, y_te_seg = X_test[mask_te],  y_test[mask_te]
    
    model = build_model_custom(X_train.shape[1], **ARCH)
    history = model.fit(
        x_tr_seg, y_tr_seg,
        epochs=1000, batch_size=32, validation_split=0.2, verbose=0,
        callbacks=[callbacks.EarlyStopping(patience=20, restore_best_weights=True)]
    )
    
    test_mae  = model.evaluate(x_te_seg, y_te_seg, verbose=0)[1]
    y_pred    = model.predict(x_te_seg, verbose=0).flatten()
    test_mape = np.mean(np.abs((y_te_seg - y_pred) / y_te_seg)) * 100
    rmse      = np.sqrt(np.mean((y_te_seg - y_pred) ** 2))
    
    print(f"  MAE: €{test_mae:.2f} | MAPE: {test_mape:.2f}%")
    final_deterministic_models[name] = model
    
    training_records.append({
        'segment': name,
        'price_min': seg['min'],
        'price_max': seg['max'],
        'n_train': len(y_tr_seg),
        'n_test': len(y_te_seg),
        'test_mae': round(float(test_mae), 4),
        'test_rmse': round(float(rmse), 4),
        'test_mape': round(float(test_mape), 4),
        'epochs_trained': len(history.history['loss'])
    })

pd.DataFrame(training_records).to_csv(
    f'{TRAINING_RESULTS_DIR}/deterministic_metrics.csv', index=False
)
print(f"\nResults saved to: {TRAINING_RESULTS_DIR}/deterministic_metrics.csv")

## Artifact Export

Save the 5 Keras models to our centralized artifacts directory for downstream evaluation mapping.

In [ ]:
ARTIFACTS_DIR = '../../artifacts'
DET_DIR       = os.path.join(ARTIFACTS_DIR, 'deterministic')
os.makedirs(DET_DIR, exist_ok=True)

for name, model in final_deterministic_models.items():
    model_path = os.path.join(DET_DIR, f'segment_{name}.keras')
    model.save(model_path)

print(f"Exported {len(final_deterministic_models)} segment models to: {DET_DIR}")